# Neo RX focused-v2 — cinco hallazgos, multietiqueta

Entrena enfisema, edema, derrame pleural, masa pulmonar y neumotórax. No fuerza un único diagnóstico. Ejecuta primero el piloto y usa una sesión/salida limpia para el entrenamiento completo.

In [ ]:
from pathlib import Path
import json, os, platform, shutil, subprocess, sys, yaml

REPO_URL = 'https://github.com/LinoMMJ/NEO-RX.git'
GIT_REF = 'main'  # fija aquí el tag de focused-v2 cuando se publique
ROOT = Path('/kaggle/working/neorx-focused5')
if not ROOT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', GIT_REF, REPO_URL, str(ROOT)], check=True)
BACKEND = ROOT / 'backend'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(BACKEND / 'training/requirements.kaggle.txt')], check=True)
os.chdir(BACKEND)
sys.path.insert(0, str(BACKEND))
print('Código:', ROOT)

In [ ]:
import torch
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Activa Accelerator > GPU')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

In [ ]:
candidates = [p for p in Path('/kaggle/input').iterdir() if any(p.rglob('Data_Entry_2017.csv'))]
if len(candidates) != 1:
    raise RuntimeError(f'Se esperaba exactamente un dataset NIH; encontrados: {candidates}')
DATA_ROOT = candidates[0]
WORK = Path('/kaggle/working')
MANIFESTS = WORK / 'manifests-focused5'
CHECKPOINTS = WORK / 'checkpoints-focused5'
RESULTS = WORK / 'results-focused5'
for directory in (MANIFESTS, CHECKPOINTS, RESULTS):
    directory.mkdir(parents=True, exist_ok=True)
with open(BACKEND / 'training/config.focused5.kaggle.yaml', encoding='utf-8') as handle:
    config = yaml.safe_load(handle)
config['data']['data_dir'] = str(DATA_ROOT)
config['data']['manifest'] = str(MANIFESTS / 'clean_manifest.csv')
config['data']['splits_json'] = str(MANIFESTS / 'splits.json')
config['model']['checkpoint_dir'] = str(CHECKPOINTS)
config['train']['log_dir'] = str(WORK / 'runs-focused5')
RUNTIME_CONFIG = WORK / 'config.focused5.runtime.yaml'
with open(RUNTIME_CONFIG, 'w', encoding='utf-8') as handle:
    yaml.safe_dump(config, handle, allow_unicode=True, sort_keys=False)
print('Objetivos:', config['target_labels'])
print('Dataset:', DATA_ROOT)

In [ ]:
if not (MANIFESTS / 'clean_manifest.csv').exists():
    subprocess.run([sys.executable, '-m', 'training.validate_dataset', '--input-root', str(DATA_ROOT), '--output-dir', str(MANIFESTS), '--workers', '4'], check=True)
else:
    print('Se reutiliza el manifiesto auditado.')
subprocess.run([sys.executable, '-m', 'training.create_splits', '--manifest', str(MANIFESTS / 'clean_manifest.csv'), '--output-dir', str(MANIFESTS), '--config', str(RUNTIME_CONFIG), '--min-positives-per-label', '25', '--seed', '42'], check=True)
summary = json.loads((MANIFESTS / 'split_summary.json').read_text(encoding='utf-8'))
print(json.dumps(summary, indent=2, ensure_ascii=False))
assert summary['patient_overlap'] == 0

## Piloto y entrenamiento completo

El piloto comprueba memoria, formas y checkpoints; sus métricas no son resultados. Para `full`, empieza sin checkpoints del piloto o adjunta una salida completa para reanudar.

In [ ]:
MODE = 'pilot'  # cambia a 'full' en una ejecución limpia
RESUME = None   # ruta a last.pt de la misma configuración, si corresponde
command = [sys.executable, '-m', 'training.train', '--config', str(RUNTIME_CONFIG), '--device', 'cuda']
if MODE == 'pilot':
    command += ['--max-samples', '2000']
if RESUME:
    command += ['--resume', RESUME]
print('Ejecutando:', ' '.join(command))
subprocess.run(command, check=True)

In [ ]:
FINAL = CHECKPOINTS / 'neorx_focused5_resnet50.pt'
if MODE != 'full':
    print('Evaluación final omitida: el piloto no produce métricas válidas.')
elif not FINAL.exists():
    raise FileNotFoundError(FINAL)
else:
    subprocess.run([sys.executable, '-m', 'training.evaluate', '--config', str(RUNTIME_CONFIG), '--checkpoint', str(FINAL), '--device', 'cuda', '--output-dir', str(RESULTS), '--plot'], check=True)
    decision = json.loads((RESULTS / 'decision_summary.json').read_text(encoding='utf-8'))
    print(json.dumps(decision, indent=2, ensure_ascii=False))

In [ ]:
archive = shutil.make_archive('/kaggle/working/neorx-focused5-checkpoints', 'zip', str(CHECKPOINTS))
print('Descarga:', archive)
print('Conserva también manifests-focused5/, results-focused5/ y runs-focused5/.')